# CƠ SỞ PHÂN TÍCH DỮ LIỆU LỚN DÙNG PYSPARK

## Notebook thực hành
Notebook này đi cùng handout Word. Đặt notebook ở thư mục gốc của bộ tài liệu, sao cho dữ liệu nằm trong `Datasets/`.

> **Lưu ý:** Nếu máy chưa cài PySpark, có thể chạy ô cài đặt bên dưới một lần. Trên Google Colab, thường cần cài `pyspark`.

In [ ]:
# Bỏ dấu # nếu cần cài đặt
# !pip install -q pyspark

## 0. Khởi tạo môi trường

In [ ]:
from pathlib import Path
import sys, re, string

DATA_DIR = Path('Datasets')
print('Python:', sys.version.split()[0])
print('Data dir exists:', DATA_DIR.exists())

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName('BigDataFundamentals')
         .master('local[*]')
         .getOrCreate())
sc = spark.sparkContext
print('Spark version:', spark.version)
print('Master:', sc.master)
print('Default parallelism:', sc.defaultParallelism)
print('App name:', sc.appName)

# BÀI 1. Làm quen với Spark và RDD

## 1.1 Parallel Collection

In [ ]:
numbers = list(range(1, 101))
numbers_rdd = sc.parallelize(numbers)
print('Count:', numbers_rdd.count())
print('First 10:', numbers_rdd.take(10))
print('Sum:', numbers_rdd.sum())
print('Min:', numbers_rdd.min(), 'Max:', numbers_rdd.max())

### Bài tập
1. Tạo RDD bình phương.
2. Lọc số chia hết cho 3.
3. Lọc số chia hết cho 5.
4. Lọc số chia hết cho cả 3 và 5.

In [ ]:
# TODO: Sinh viên hoàn thiện
squares = numbers_rdd.map(lambda x: x**2)
div3 = numbers_rdd.filter(lambda x: x % 3 == 0)
div5 = numbers_rdd.filter(lambda x: x % 5 == 0)
div15 = numbers_rdd.filter(lambda x: x % 15 == 0)
print(squares.take(10))
print(div3.take(10))
print(div5.take(10))
print(div15.collect())

## 1.2 Transformation, Action và Lazy Evaluation

In [ ]:
rdd = sc.parallelize(range(1, 1_000_000))
result = rdd.filter(lambda x: x % 2 == 0).map(lambda x: x * x)
# Transformation chưa buộc Spark tính toán ngay.
result.take(10)

**Câu hỏi:** `filter`, `map`, `take` thuộc loại nào? Vì sao Spark dùng lazy evaluation?

# BÀI 2. Xử lý văn bản với RDD

## 2.1 Đọc Complete Shakespeare

In [ ]:
shakespeare_path = DATA_DIR / 'Complete Shakespeare.txt'
text_rdd = sc.textFile(str(shakespeare_path))
print('Lines:', text_rdd.count())
print('Partitions:', text_rdd.getNumPartitions())
for line in text_rdd.take(5):
    print(line)

## 2.2 Word Count cơ bản

In [ ]:
words = text_rdd.flatMap(lambda line: line.split())
word_counts = (words
               .map(lambda w: (w, 1))
               .reduceByKey(lambda a, b: a + b))
word_counts.take(10)

In [ ]:
top20_raw = word_counts.takeOrdered(20, key=lambda x: -x[1])
top20_raw

## 2.3 Làm sạch văn bản

In [ ]:
stopwords = set(['the','and','of','to','a','in','that','is','it','for','as','with','his','he','be','this','not','but','you'])

def clean_line(line):
    line = line.lower()
    line = re.sub(r'[^a-z\s]', ' ', line)
    return [w for w in line.split() if len(w) >= 3 and w not in stopwords]

clean_words = text_rdd.flatMap(clean_line)
clean_counts = clean_words.map(lambda w: (w, 1)).reduceByKey(lambda a,b: a+b)
clean_counts.takeOrdered(20, key=lambda x: -x[1])

## 2.4 Phân tích chiều dài từ

In [ ]:
length_groups = (clean_words
    .map(lambda w: ('1-4' if len(w) <= 4 else '5-8' if len(w) <= 8 else '>=9', 1))
    .reduceByKey(lambda a,b: a+b)
    .collectAsMap())
length_groups

## 2.5 Bài tập mở rộng
- So sánh `groupByKey()` và `reduceByKey()`.
- Thử `distinct()`, `sortByKey()`, `repartition()` và `coalesce()`.

In [ ]:
# TODO: thử các transformation nâng cao tại đây

# BÀI 3. DataFrame và Spark SQL với FIFA 2018

## 3.1 Đọc dữ liệu

In [ ]:
fifa_path = DATA_DIR / 'Fifa2018_dataset.csv'
df = (spark.read
      .option('header', True)
      .option('inferSchema', True)
      .csv(str(fifa_path)))
print('Rows:', df.count())
print('Columns:', len(df.columns))
df.printSchema()
df.show(5, truncate=False)

## 3.2 Missing values

In [ ]:
missing_exprs = [F.sum(F.col(c).isNull().cast('int')).alias(c) for c in df.columns]
missing = df.select(missing_exprs)
missing.show(truncate=False)

## 3.3 Xác định các cột quan trọng

In [ ]:
print(df.columns)
# Một số phiên bản dataset có tên cột khác nhau.
# Chọn các cột tồn tại trong danh sách mong muốn:
wanted = ['Name','Age','Nationality','Overall','Potential','Club']
selected = [c for c in wanted if c in df.columns]
print('Selected columns:', selected)
df.select(*selected).show(10, truncate=False)

## 3.4 Filter và OrderBy

In [ ]:
# Các ô dưới đây chỉ chạy khi dataset có các cột tương ứng.
if 'Overall' in df.columns:
    df.filter(F.col('Overall') > 85).orderBy(F.desc('Overall')).show(20, truncate=False)
if {'Age','Potential'}.issubset(df.columns):
    df.filter(F.col('Age') < 21).orderBy(F.desc('Potential')).show(10, truncate=False)

## 3.5 GroupBy và Aggregate

In [ ]:
if 'Nationality' in df.columns:
    df.groupBy('Nationality').count().orderBy(F.desc('count')).show(10, truncate=False)

if {'Nationality','Overall','Potential','Age'}.issubset(df.columns):
    (df.groupBy('Nationality')
       .agg(F.avg('Overall').alias('avg_overall'),
            F.avg('Potential').alias('avg_potential'),
            F.avg('Age').alias('avg_age'),
            F.count('*').alias('n'))
       .orderBy(F.desc('avg_overall'))
       .show(20, truncate=False))

## 3.6 Phân tích câu lạc bộ

In [ ]:
if {'Club','Overall','Potential','Age'}.issubset(df.columns):
    club_stats = (df.groupBy('Club')
        .agg(F.count('*').alias('players'),
             F.avg('Overall').alias('avg_overall'),
             F.avg('Potential').alias('avg_potential'),
             F.avg('Age').alias('avg_age')))
    club_stats.orderBy(F.desc('players')).show(10, truncate=False)

## 3.7 Spark SQL

In [ ]:
df.createOrReplaceTempView('fifa')
spark.sql('SELECT * FROM fifa LIMIT 10').show(truncate=False)

In [ ]:
# Ví dụ động: chỉ chạy nếu cột tồn tại
if 'Overall' in df.columns:
    spark.sql('SELECT * FROM fifa ORDER BY Overall DESC LIMIT 10').show(truncate=False)
if {'Nationality','Overall'}.issubset(df.columns):
    spark.sql('''
        SELECT Nationality, AVG(Overall) AS avg_overall, COUNT(*) AS n
        FROM fifa
        GROUP BY Nationality
        ORDER BY avg_overall DESC
        LIMIT 10
    ''').show(truncate=False)

### Bài tập Spark SQL
1. Viết truy vấn tìm cầu thủ `Age < 21` và `Potential >= 85`.
2. Tìm 10 quốc gia có nhiều cầu thủ nhất.
3. So sánh kết quả với DataFrame API.

In [ ]:
# TODO: viết truy vấn SQL ở đây

# BÀI 4. Machine Learning với PySpark

## 4A. K-Means trên 5000_points.txt

In [ ]:
points_path = DATA_DIR / '5000_points.txt'
# Tập này thường là 2 cột số, phân tách bởi khoảng trắng/tab.
raw_points = spark.read.text(str(points_path))
points = (raw_points
    .select(F.split(F.trim('value'), r'\s+').alias('v'))
    .filter(F.size('v') >= 2)
    .select(F.col('v')[0].cast('double').alias('x'),
            F.col('v')[1].cast('double').alias('y'))
    .na.drop())
points.show(5)
print('Rows:', points.count())

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

assembler = VectorAssembler(inputCols=['x','y'], outputCol='features')
points_vec = assembler.transform(points)
evaluator = ClusteringEvaluator(featuresCol='features', predictionCol='prediction', metricName='silhouette')

results = []
for k in range(2, 21):
    model = KMeans(k=k, seed=42, featuresCol='features').fit(points_vec)
    pred = model.transform(points_vec)
    score = evaluator.evaluate(pred)
    results.append((k, score))
results

In [ ]:
best_k, best_score = max(results, key=lambda x: x[1])
print('Best k:', best_k, 'Silhouette:', best_score)
best_model = KMeans(k=best_k, seed=42, featuresCol='features').fit(points_vec)
predictions = best_model.transform(points_vec)

### Trực quan hóa
Chỉ chuyển số lượng dữ liệu cần vẽ sang Pandas. Với Big Data thật, nên `sample()` hoặc `limit()` trước `toPandas()`.

In [ ]:
import matplotlib.pyplot as plt
plot_df = predictions.select('x','y','prediction').limit(10000).toPandas()
plt.figure(figsize=(7,5))
plt.scatter(plot_df['x'], plot_df['y'], c=plot_df['prediction'], s=8)
plt.xlabel('x'); plt.ylabel('y'); plt.title(f'K-Means, k={best_k}')
plt.show()

## 4B. Phân loại Spam/Ham

In [ ]:
spam_rdd = sc.textFile(str(DATA_DIR / 'spam.txt')).map(lambda x: (x, 1.0))
ham_rdd  = sc.textFile(str(DATA_DIR / 'ham.txt')).map(lambda x: (x, 0.0))
text_df = spark.createDataFrame(spam_rdd.union(ham_rdd), ['text','label'])
text_df.groupBy('label').count().show()
text_df.show(5, truncate=80)

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

train_df, test_df = text_df.randomSplit([0.8, 0.2], seed=42)

tokenizer = Tokenizer(inputCol='text', outputCol='tokens')
remover = StopWordsRemover(inputCol='tokens', outputCol='filtered')
hashing = HashingTF(inputCol='filtered', outputCol='tf', numFeatures=1<<14)
idf = IDF(inputCol='tf', outputCol='features')
lr = LogisticRegression(featuresCol='features', labelCol='label', maxIter=50)

pipeline = Pipeline(stages=[tokenizer, remover, hashing, idf, lr])
model = pipeline.fit(train_df)
pred = model.transform(test_df)
pred.select('text','label','prediction','probability').show(10, truncate=70)

In [ ]:
acc_eval = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='accuracy')
f1_eval  = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='f1')
prec_eval = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='weightedPrecision')
rec_eval = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='weightedRecall')
auc_eval = BinaryClassificationEvaluator(labelCol='label', rawPredictionCol='rawPrediction', metricName='areaUnderROC')

print('Accuracy :', acc_eval.evaluate(pred))
print('Precision:', prec_eval.evaluate(pred))
print('Recall   :', rec_eval.evaluate(pred))
print('F1       :', f1_eval.evaluate(pred))
print('ROC-AUC  :', auc_eval.evaluate(pred))

**Câu hỏi:** Accuracy có luôn là metric tốt nhất cho phát hiện spam không? Khi dữ liệu mất cân bằng nên quan tâm thêm những metric nào?

# BÀI 5. Bài tập bổ sung với people.csv và Movie ratings.csv

## 5.1 people.csv

In [ ]:
people_path = DATA_DIR / 'people.csv'
people = spark.read.option('header', True).option('inferSchema', True).csv(str(people_path))
people.show()
people.printSchema()

**Yêu cầu:** tự đặt ít nhất 3 câu hỏi phân tích phù hợp với các cột thực tế của `people.csv`.

## 5.2 Movie ratings.csv

In [ ]:
ratings_path = DATA_DIR / 'Movie ratings.csv'
ratings = spark.read.option('header', True).option('inferSchema', True).csv(str(ratings_path))
ratings.show(5, truncate=False)
ratings.printSchema()

**Yêu cầu:**
- Kiểm tra schema và missing values.
- Tìm nhóm/phim/người dùng có số lượng rating lớn nhất tùy theo cấu trúc file.
- Tính rating trung bình theo một chiều dữ liệu phù hợp.
- Viết ít nhất 2 truy vấn Spark SQL.

# BÀI MỞ RỘNG

Chọn dataset tối thiểu **100.000 dòng**, thực hiện:
1. Data Understanding.
2. Data Cleaning.
3. Ít nhất 6 thao tác DataFrame.
4. Ít nhất 5 truy vấn Spark SQL.
5. Ít nhất 5 câu hỏi phân tích.
6. Một mô hình ML bằng `pyspark.ml`.
7. Kết luận và hạn chế.

## Checklist nộp bài
- [ ] Notebook chạy từ đầu đến cuối.
- [ ] Không dùng `collect()`/`toPandas()` cho toàn bộ Big Data.
- [ ] Có giải thích kết quả, không chỉ có code.
- [ ] Có Spark SQL và DataFrame API.
- [ ] Có ít nhất một mô hình ML.
- [ ] Có phần kết luận.